In [0]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import logging
from datetime import datetime
from typing import List, Dict, Any, Optional
import re
import os

In [0]:
def setup_logger(filename):
    if not os.path.exists(filename):
        with open(filename, 'w') as f:
            pass

    logger = logging.getLogger("WebScrapLogger")
    logger.setLevel(logging.DEBUG)

    if logger.hasHandlers():
        logger.handlers.clear()

    file_handler = logging.FileHandler(filename,mode="a")
    console_handler = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)
    console_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)
    logger.info("Logger setup completed")
    return logger

logger = setup_logger("/tmp/weather_scraper_log.log")

In [0]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import logging
from datetime import datetime
from typing import List, Dict, Any, Optional
import re
import os

class WeatherDataScraper:
    """Web scraper class for weather data collection from National Weather Service"""
    
    def __init__(self, cities_to_scrape: List[str]):
        self.base_url = "https://forecast.weather.gov"
        self.cities = cities_to_scrape
        self.city_urls = {
            "seattle": "/MapClick.php?lat=47.6062&lon=-122.3321",
            "new_york": "/MapClick.php?lat=40.7128&lon=-74.0060",
            "chicago": "/MapClick.php?lat=41.8781&lon=-87.6298",
            "los_angeles": "/MapClick.php?lat=34.0522&lon=-118.2437",
            "miami": "/MapClick.php?lat=25.7617&lon=-80.1918",
            "denver": "/MapClick.php?lat=39.7392&lon=-104.9903",
            "phoenix": "/MapClick.php?lat=33.4484&lon=-112.0740",
            "boston": "/MapClick.php?lat=42.3601&lon=-71.0589",
            "houston": "/MapClick.php?lat=29.7604&lon=-95.3698",
            "atlanta": "/MapClick.php?lat=33.7490&lon=-84.3880",
        }
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        }
        self.data = []
        
    def _get_page_content(self, url: str) -> Optional[BeautifulSoup]:
        try:
            time.sleep(random.uniform(1.0, 2.0))
            response = requests.get(url, headers=self.headers)
            if response.status_code == 200:
                logger.info(f"Successfully fetched page: {url}")
                return BeautifulSoup(response.text, 'html.parser')
            else:
                logger.error(f"Failed to fetch page: {url}, Status code: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            logger.error(f"Request error while fetching {url}: {e}")
            return None
    
    def _extract_weather_data(self, soup: BeautifulSoup, city: str) -> List[Dict[str, Any]]:
        weather_data = []
        try:
            forecast_container = soup.find('div', id='seven-day-forecast')
            if not forecast_container:
                logger.error(f"Could not find 7-day forecast container for {city}")
                return weather_data
            forecast_items = forecast_container.select('div.tombstone-container')
            for item in forecast_items:
                try:
                    period_name = item.find(class_='period-name').get_text(strip=True)
                    short_desc = item.find(class_='short-desc').get_text(strip=True)
                    temp_elem = item.find(class_='temp')
                    temp_text = temp_elem.get_text(strip=True) if temp_elem else None
                    temp_match = re.search(r'(High|Low):?\s*(\d+)', temp_text or '')
                    temperature = int(temp_match.group(2)) if temp_match else None
                    is_high_temp = temp_match.group(1) == 'High' if temp_match else None
                    img = item.find('img')
                    detailed_forecast = img['title'] if img and 'title' in img.attrs else short_desc
                    precip_match = re.search(r'(\d+)% chance', detailed_forecast, re.IGNORECASE)
                    precip_chance = int(precip_match.group(1)) if precip_match else 0
                    forecast_item = {
                        'city': city,
                        'period': period_name,
                        'temperature': temperature,
                        'is_high_temp': is_high_temp,
                        'short_forecast': short_desc,
                        'detailed_forecast': detailed_forecast,
                        'precipitation_chance': precip_chance,
                        'scraped_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    }
                    weather_data.append(forecast_item)
                    logger.debug(f"Extracted forecast for {city} - {period_name}")
                except (AttributeError, ValueError) as e:
                    logger.warning(f"Error extracting forecast data for {city}: {e}")
                    continue
        except Exception as e:
            logger.error(f"Error processing weather data for {city}: {e}")
        return weather_data
    
    def scrape_weather(self) -> pd.DataFrame:
        all_weather_data = []
        for city in self.cities:
            if city.lower() in self.city_urls:
                city_url = f"{self.base_url}{self.city_urls[city.lower()]}"
                soup = self._get_page_content(city_url)
                if soup:
                    city_weather_data = self._extract_weather_data(soup, city)
                    all_weather_data.extend(city_weather_data)
                    logger.info(f"Scraped {len(city_weather_data)} forecast periods for {city}")
                else:
                    logger.warning(f"Failed to scrape data for {city}")
        weather_df = pd.DataFrame(all_weather_data)
        logger.info(f"Data scraping complete. Shape: {weather_df.shape}")
        return weather_df
    
    def save_data(self, df: pd.DataFrame, file_path: str) -> None:
        try:
            import os
            os.makedirs(os.path.dirname(file_path), exist_ok=True)
            df.to_csv(file_path, index=False)
            logger.info(f"Data successfully saved to {file_path}")
        except Exception as e:
            logger.error(f"Error saving data: {e}")

def main():
    try:
        cities = ["seattle", "new_york", "chicago", "los_angeles", "miami"]
        logger.info("Starting web scraping process")
        scraper = WeatherDataScraper(cities)
        raw_data = scraper.scrape_weather()
        if not raw_data.empty:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            file_path = f"scraped_data/weather_data_{timestamp}.csv"
            scraper.save_data(raw_data, file_path)
            # Print summary statistics
            logger.info("\n----- Data Summary -----")
            logger.info(f"Total forecast periods scraped: {len(raw_data)}")
            logger.info(f"Cities scraped: {raw_data['city'].nunique()}")
            logger.info(f"Average temperature: {raw_data['temperature'].mean():.1f}°F")
            logger.info(f"Temperature range: {raw_data['temperature'].min()}°F - {raw_data['temperature'].max()}°F")
            # Display DataFrame in Databricks
            display(raw_data)
        else:
            logger.warning("No data was scraped")
    except Exception as e:
        logger.error(f"An error occurred in the main execution: {e}")
        
if __name__ == "__main__":
    main()


In [0]:
# Read and display the contents of the weather_scraper_log.log file for debugging

def read_log_file(file_path):
    try:
        with open(file_path, 'r') as file:
            log_contents = file.read()
            print("Log File Contents:\n")
            print(log_contents)
    except FileNotFoundError:
        print(f"Error: The file {file_path} was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

# Specify the path to the log file
log_file_path = '/tmp/weather_scraper_log.log'
read_log_file(log_file_path)